# Notebook 02 — Feature Engineering

**Goal:** Compute all behavioural features from the enriched scrobble data and produce a user-level feature matrix for clustering.

**Inputs** (from `data/processed/`):
- `scrobbles_updated.parquet`
- `profiles.parquet`
- `artist_genres.parquet`
- `audio_features.parquet`

**Outputs:**
- `data/processed/user_features.parquet` — one row per user, all features

**Features computed:**

| Group | Features |
|---|---|
| Artist diversity | unique_artists, artist_entropy, artist_concentration_20 |
| Genre diversity | unique_genres, genre_entropy, genre_concentration_5, avg_genre_tags_per_play |
| Engagement | total_scrobbles, unique_tracks, track_replay_rate, avg_tracks_per_session, session_count |
| Discovery | discovery_velocity_30d/90d, novelty_ratio, top_artist_play_share |
| Temporal | temporal_hour/dow/month_entropy, morning/evening/weekend_ratio, temporal_stability_pc1..5 |
| Audio profile | mean danceability, energy, valence, tempo, acousticness, instrumentalness |

In [ ]:
import sys
import logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load all enriched data
scrobbles    = pd.read_parquet('../data/processed/scrobbles_updated.parquet')
profiles     = pd.read_parquet('../data/processed/profiles.parquet')
artist_genres = pd.read_parquet('../data/processed/artist_genres.parquet')
audio_features = pd.read_parquet('../data/processed/audio_features.parquet')

print(f'Scrobbles: {len(scrobbles):,} rows, {scrobbles["userid"].nunique()} users')
print(f'Artist genres: {len(artist_genres):,} artists, {artist_genres["spotify_artist_id"].notna().sum():,} matched')
print(f'Audio features: {len(audio_features):,} tracks')

## 1. Artist Diversity Features

In [ ]:
from src.features.diversity import compute_artist_diversity

artist_div = compute_artist_diversity(scrobbles, top_n=20)
print(f'Artist diversity features: {artist_div.shape}')
artist_div.describe()

In [ ]:
# Artist diversity — three individual EPS figures (one per metric)
import matplotlib.pyplot as plt
from pathlib import Path

figures_dir = Path('../outputs/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

# ── Unique Artists Per User ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
artist_div['unique_artists'].hist(ax=ax, bins=30, color='#636EFA', edgecolor='white')
ax.set_title('Unique Artists Per User', fontsize=13, fontweight='bold')
ax.set_xlabel('Unique Artist Count')
ax.set_ylabel('Number Of Users')
plt.tight_layout()
plt.savefig(figures_dir / 'artist_diversity_unique_artists.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'artist_diversity_unique_artists.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Artist Shannon Entropy ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
artist_div['artist_entropy'].hist(ax=ax, bins=30, color='#EF553B', edgecolor='white')
ax.set_title('Artist Shannon Entropy', fontsize=13, fontweight='bold')
ax.set_xlabel('Entropy (Bits)')
ax.set_ylabel('Number Of Users')
plt.tight_layout()
plt.savefig(figures_dir / 'artist_diversity_entropy.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'artist_diversity_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Top-20 Artist Concentration ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
artist_div['artist_concentration_20'].hist(ax=ax, bins=30, color='#00CC96', edgecolor='white')
ax.set_title('Top-20 Artist Concentration', fontsize=13, fontweight='bold')
ax.set_xlabel('Fraction Of Plays From Top 20 Artists')
ax.set_ylabel('Number Of Users')
plt.tight_layout()
plt.savefig(figures_dir / 'artist_diversity_concentration.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'artist_diversity_concentration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Artist diversity features cover {len(artist_div)} users (all scrobble users represented).")

## 2. Genre Diversity Features

In [ ]:
from src.features.diversity import compute_genre_diversity

genre_div = compute_genre_diversity(scrobbles, artist_genres, top_n=5)
print(f'Genre diversity features: {genre_div.shape}  ({genre_div.index.nunique()} users)')

n_with_genres = int((genre_div['unique_genres'] > 0).sum())
n_total       = len(genre_div)
print(f'Users with ≥1 Spotify genre match : {n_with_genres} / {n_total} ({n_with_genres / n_total * 100:.1f}%)')
print(f'Users with no genre data (zero-filled): {n_total - n_with_genres}')
print()
genre_div.describe()

## 3. Temporal Features

Includes PCA-compressed hourly listening profiles to capture **listening pattern stability** while avoiding multicollinearity across 24 hour-of-day columns.

In [ ]:
from src.features.temporal import compute_temporal_features

temporal = compute_temporal_features(scrobbles, pca_components=5)
print(f'Temporal features: {temporal.shape}')
temporal.describe()

In [ ]:
# Temporal features — three individual EPS figures
import matplotlib.pyplot as plt
from pathlib import Path

figures_dir = Path('../outputs/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

# ── Hour-Of-Day Entropy ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
temporal['temporal_hour_entropy'].hist(ax=ax, bins=20, color='#636EFA', edgecolor='white')
ax.set_title('Hour-Of-Day Entropy', fontsize=13, fontweight='bold')
ax.set_xlabel('Entropy (Bits)')
ax.set_ylabel('Number Of Users')
plt.tight_layout()
plt.savefig(figures_dir / 'temporal_hour_entropy.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'temporal_hour_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Weekend Listen Ratio ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
temporal['weekend_ratio'].hist(ax=ax, bins=20, color='#EF553B', edgecolor='white')
ax.set_title('Weekend Listen Ratio', fontsize=13, fontweight='bold')
ax.set_xlabel('Fraction Of Plays On Weekends')
ax.set_ylabel('Number Of Users')
plt.tight_layout()
plt.savefig(figures_dir / 'temporal_weekend_ratio.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'temporal_weekend_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Average Daily Plays ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
temporal['avg_daily_plays'].hist(ax=ax, bins=30, color='#00CC96', edgecolor='white')
ax.set_title('Average Daily Plays', fontsize=13, fontweight='bold')
ax.set_xlabel('Avg Plays Per Active Day')
ax.set_ylabel('Number Of Users')
plt.tight_layout()
plt.savefig(figures_dir / 'temporal_avg_daily_plays.eps', format='eps', bbox_inches='tight')
plt.savefig(figures_dir / 'temporal_avg_daily_plays.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Temporal features cover {len(temporal)} users (all scrobble users represented).")

## 4. Engagement & Discovery Features

In [ ]:
from src.features.engagement import compute_engagement_features

engagement = compute_engagement_features(scrobbles, session_gap_minutes=30)
print(f'Engagement features: {engagement.shape}')
engagement.describe()

In [ ]:
# Engagement features — six individual EPS figures
import matplotlib.pyplot as plt
from pathlib import Path

figures_dir = Path('../outputs/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

engagement_plots = [
    ('track_replay_rate',        'Track Replay Rate',        'Plays / Unique Tracks'),
    ('avg_tracks_per_session',   'Avg Tracks Per Session',   'Tracks'),
    ('discovery_velocity_30d',   'Discovery Velocity (30 Days)', 'New Artists / Day'),
    ('novelty_ratio',            'Novelty Ratio',            'Fraction Of Plays From New Artists'),
    ('top_artist_play_share',    'Top Artist Play Share',    'Fraction Of Plays'),
    ('session_count',            'Session Count',            'Number Of Sessions'),
]

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']

for (col, title, xlabel), color in zip(engagement_plots, colors):
    if col not in engagement.columns:
        print(f"Column '{col}' not found — skipping.")
        continue
    fig, ax = plt.subplots(figsize=(6, 4))
    engagement[col].hist(ax=ax, bins=25, color=color, edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Number Of Users')
    plt.tight_layout()
    slug = col.replace('_', '-')
    plt.savefig(figures_dir / f'engagement_{slug}.eps', format='eps', bbox_inches='tight')
    plt.savefig(figures_dir / f'engagement_{slug}.png', dpi=150, bbox_inches='tight')
    plt.show()

print(f"Engagement features cover {len(engagement)} users (all scrobble users represented).")

## 5. Audio Feature Profile

In [ ]:
from src.features.engagement import compute_audio_feature_profile

audio_profile = compute_audio_feature_profile(scrobbles, audio_features)
print(f'Audio feature profile: {audio_profile.shape}')
audio_profile.head()

## 6. Build Final Feature Matrix

In [ ]:
from src.features.builder import build_feature_matrix

feature_matrix = build_feature_matrix(
    scrobbles=scrobbles,
    artist_genres=artist_genres,
    audio_features=audio_features,
    profiles=profiles,
    config_path='../configs/config.yaml',
    save_path='../data/processed/user_features.parquet',
)

print(f'Feature matrix: {feature_matrix.shape[0]} users × {feature_matrix.shape[1]} features')
feature_matrix.head()

## 7. Correlation Analysis & Multicollinearity Check

In [ ]:
# Select numeric clustering features (exclude demographic descriptors)
clustering_features = feature_matrix.select_dtypes(include=[np.number]).drop(
    columns=['age'], errors='ignore'
)

corr = clustering_features.corr()

fig, ax = plt.subplots(figsize=(20, 16))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='RdBu', center=0, vmin=-1, vmax=1,
    annot=False, linewidths=0.3, ax=ax,
)
ax.set_title('Feature Correlation Matrix (Clustering Features)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.eps', format='eps', bbox_inches='tight')
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Pairs with |r| > 0.75
high_corr = (
    corr.abs().where(np.tril(np.ones(corr.shape), k=-1).astype(bool))
    .stack().reset_index()
    .rename(columns={0: 'correlation', 'level_0': 'feat_a', 'level_1': 'feat_b'})
    .query('correlation > 0.75')
    .sort_values('correlation', ascending=False)
)
print(f'Highly correlated pairs (|r| > 0.75): {len(high_corr)}')
print(high_corr.to_string())

> Note: Remaining multicollinearity is handled at the clustering stage via PCA dimensionality reduction before fitting KMeans/HDBSCAN.

Proceed to **Notebook 03** for clustering.